In [22]:
from tqdm import tqdm
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms

In [2]:
# Helper Functions
def generate_hypervector(d):
    """Generates a random D-dimensional bipolar vector."""
    return np.random.choice([-1, 1], size=d)

def bind(a, b):
    """Binding operation using element-wise multiplication (XOR in bipolar)."""
    return a * b

def bundle(vectors):
    """Bundling operation: sum and then take the sign (majority vote)."""
    summed = np.sum(vectors, axis=0)
    # Handle ties with random choice or 1
    return np.where(summed >= 0, 1, -1)

def cosine_similarity(a, b):
    """Measures how similar two hypervectors are."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [3]:
# Encoding Function
def encode_image(img_flat):
    """
    img_flat: flattened grayscale image (1024 pixels)
    Returns: A single D-dimensional hypervector representing the image
    """
    pixel_vectors = []
    for pos_idx, val in enumerate(img_flat):
        # Bind the POSITION to the VALUE
        v_pos = position_codebook[pos_idx]
        v_val = value_codebook[int(val)]
        pixel_vectors.append(bind(v_pos, v_val))

    # Bundle all pixels into one image vector
    return bundle(pixel_vectors)

In [4]:
# 1. Hyperparameters
D = 10000        # Hypervector dimension
NUM_CLASSES = 10 # CIFAR-10 classes
IMG_SIZE = 32    # CIFAR-10 images are 32x32

In [5]:
# Initialize Codebooks (The "Memory")
# One random vector for each pixel position (32 * 32 = 1024)
position_codebook = np.random.choice([-1, 1], size=(IMG_SIZE * IMG_SIZE, D))
# One random vector for each pixel intensity (0-255)
value_codebook = np.random.choice([-1, 1], size=(256, D))

In [6]:
# (Loading actual CIFAR-10 for a toy demo)
transform = transforms.Compose([transforms.Grayscale(), transforms.ToTensor()])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

100%|██████████| 170M/170M [00:04<00:00, 40.6MB/s]


In [7]:
# Let's "train" on the first 1000 images to create class prototypes
class_prototypes = np.zeros((NUM_CLASSES, D))
print("Training (Bundling Prototypes)...")
for i in range(1000):
    img, label = trainset[i]
    img_flat = (img.numpy().flatten() * 255).astype(int)
    img_hv = encode_image(img_flat)
    class_prototypes[label] += img_hv

Training (Bundling Prototypes)...


In [8]:
# Finalize prototypes by taking the sign
class_prototypes = np.where(class_prototypes >= 0, 1, -1)

This "Vanilla" HDC code on CIFAR-10 will likely give you ~20-30% accuracy out of the box because raw pixel-to-position mapping is quite sensitive to shifts. In 2026, state-of-the-art HDC models use Convolutional Encoders first to extract features, then use HDC for the high-level reasoning and classification.

In [9]:
# Test on a single image
print("Testing...")
test_img, test_label = trainset[1001] # Use an unseen image
test_flat = (test_img.numpy().flatten() * 255).astype(int)
query_hv = encode_image(test_flat)

# Compare query to all 10 prototypes
similarities = [cosine_similarity(query_hv, class_prototypes[c]) for c in range(NUM_CLASSES)]
prediction = np.argmax(similarities)

print(f"True Label: {test_label}, Predicted Label: {prediction}")

Testing...
True Label: 4, Predicted Label: 7


Let's improve it a little bit

In [10]:
import torch.nn as nn
from torchvision import models, datasets

In [11]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
# Load Pre-trained ResNet & Modify for HDC
# (1) We take a pre-trained resnet18, but we remove the last fully connected layer

resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# (2) Remove the final classification layer
# ResNet18's last layer before 'fc' is 512-dimensional
feature_dim = resnet.fc.in_features
resnet.fc = nn.Identity() # Effectively removes the layer
resnet.to(DEVICE).eval()

# (3) Project to Hyper-Space: We take the output of the global average pooling layer
# (512 dimensions) and project it into our 10,000-dimensional HDC space.
# add Projection Layer (Random but fixed)
# This maps the 512 ResNet features into the 10,000 HDC dimensions
projection = nn.Linear(feature_dim, D, bias=False).to(DEVICE)
with torch.no_grad():
    # Standard HDC trick: use a random Gaussian matrix and take the sign
    projection.weight.data = torch.randn(D, feature_dim).to(DEVICE)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 191MB/s]


In [13]:
# Storage for Class Prototypes
class_prototypes = torch.zeros(NUM_CLASSES, D).to(DEVICE)

In [14]:
# Data Loading (CIFAR-10)
# We are not reusing the previous dataloader, because it uses 32x32 images
transform = transforms.Compose([
    transforms.Resize(224), # ResNet likes 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_loader = torch.utils.data.DataLoader(
    datasets.CIFAR10('./data', train=True, download=True, transform=transform),
    batch_size=64, shuffle=False
)

In [15]:
# "Training" (Single pass through the data)
print("Generating HDC Prototypes...")
with torch.no_grad():
    for images, labels in train_loader:
        images = images.to(DEVICE)
        # Extract 512 features
        features = resnet(images)
        # Project to 10,000D and binarize
        hvs = torch.sign(projection(features))

        # Bundle into prototypes
        for i in range(len(labels)):
            class_prototypes[labels[i]] += hvs[i]

# Finalize: Majority vote
class_prototypes = torch.sign(class_prototypes)
print("Prototypes created.")

Generating HDC Prototypes...
Prototypes created.


In [17]:
def classify(image_tensor, class_prototypes):
    with torch.no_grad():
        feat = resnet(image_tensor.unsqueeze(0).to(DEVICE))
        query_hv = torch.sign(projection(feat))
        # Similarity = dot product in bipolar space
        similarities = torch.matmul(class_prototypes, query_hv.T)
        return torch.argmax(similarities).item()

In [18]:
test_img, true_label = datasets.CIFAR10('./data', train=False, transform=transform)[0]
prediction = classify(test_img, class_prototypes)
print(f"Prediction: {prediction} | Actual: {true_label}")

Prediction: 3 | Actual: 3


## Online HDC Refinement.

If the accuracy is lower than you'd like, you can perform a few "retraining" passes where you subtract the query vector from the wrong prototype and add it to the right one if the model makes a mistake. This is called Online HDC Refinement.

When a sample $H_{query}$ (with true label $y$) is classified:
- If Correct: Do nothing (or update slightly if the similarity is low—this is called "Margin Learning").
- If Incorrect: Suppose the model predicted Class $W$ (Wrong) instead of Class $C$ (Correct):
  - Subtract the query from the wrong prototype: $P_{W} = P_{W} - (\eta \times H_{query})$
  - Add the query to the correct prototype: $P_{C} = P_{C} + (\eta \times H_{query})$ where $\eta$ is the learning rate.

In [19]:
learning_rate = 0.1
epochs = 5  # We now iterate over the data a few times

In [24]:
print("Starting Online Refinement...")

for epoch in range(epochs):
    correct_count = 0
    total_count = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch: {epoch+1}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        with torch.no_grad():
            features = resnet(images)
            hvs = torch.sign(projection(features)) # Encoded Hypervectors

        # Batch Refinement
        for i in range(len(labels)):
            query_hv = hvs[i]
            true_label = labels[i].item()

            # Find current best match
            similarities = torch.matmul(class_prototypes, query_hv)
            pred_label = torch.argmax(similarities).item()

            if pred_label != true_label:
                # ERROR: Update prototypes
                # 1. Subtract from the one we mistakenly picked
                class_prototypes[pred_label] -= learning_rate * query_hv
                # 2. Add to the one it actually belongs to
                class_prototypes[true_label] += learning_rate * query_hv
            else:
                correct_count += 1
            total_count += 1

    accuracy = correct_count / total_count
    print(f"Epoch {epoch+1}/{epochs} - Training Accuracy: {accuracy:.4f}")

# Final step: Binarize for fast inference (optional but standard for HDC)
final_prototypes = torch.sign(class_prototypes)

Starting Online Refinement...


Epoch: 0: 100%|██████████| 782/782 [02:15<00:00,  5.77it/s]


Epoch 1/5 - Training Accuracy: 0.8566


Epoch: 1: 100%|██████████| 782/782 [02:14<00:00,  5.82it/s]


Epoch 2/5 - Training Accuracy: 0.8858


...

Epoch 3/5 - Training Accuracy: 0.9045


Epoch: 3: 100%|██████████| 782/782 [02:15<00:00,  5.78it/s]


Epoch 4/5 - Training Accuracy: 0.9222


Epoch: 4: 100%|██████████| 782/782 [02:15<00:00,  5.76it/s]

Epoch 5/5 - Training Accuracy: 0.9335


In [25]:
test_img, true_label = datasets.CIFAR10('./data', train=False, transform=transform)[0]
prediction = classify(test_img, class_prototypes)
print(f"Prediction: {prediction} | Actual: {true_label}")

Prediction: 3 | Actual: 3
